In [ ]:
from __future__ import annotations

import argparse
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

import requests
from tqdm import tqdm

DEFAULT_PUBLIC_KEY = "YANDEX LINK"
DEFAULT_ZIP_PATH = Path("/content/input.zip")
DEFAULT_EXTRACT_DIR = Path("/content/extracted")
DEFAULT_OUTPUT_DIR = Path("/content/output_final")
DEFAULT_LOG_FILE = Path("/content/processing_log.txt")
DEFAULT_ZIP_OUT = Path("/content/2_urmi_labeled.zip")

SUPPORTED_EXTENSIONS = {".wav", ".mp3", ".flac", ".ogg", ".m4a", ".aac", ".opus"}


def strip_leading_jupyter_kernel_args(rest: list[str]) -> list[str]:
    r = list(rest)
    while len(r) >= 2 and r[0] == "-f":
        r = r[2:]
    return r


def download_yandex_public(public_key: str, dest: Path) -> None:
    api_url = "https://cloud-api.yandex.net/v1/disk/public/resources/download"
    r = requests.get(api_url, params={"public_key": public_key}, timeout=120)
    r.raise_for_status()
    href = r.json().get("href")
    if not href:
        raise RuntimeError(f"Нет href в ответе API: {r.text[:500]}")

    dest.parent.mkdir(parents=True, exist_ok=True)
    with requests.get(href, stream=True, timeout=300) as resp:
        resp.raise_for_status()
        with open(dest, "wb") as f:
            for chunk in tqdm(
                resp.iter_content(256 * 1024),
                desc="Скачивание",
                unit="chunk",
            ):
                if chunk:
                    f.write(chunk)


def find_audios_folder(base_path: Path) -> Path | None:
    for p in base_path.rglob("audios"):
        if p.is_dir():
            return p
    return None


def convert_audio(input_path: Path, output_path: Path) -> tuple[bool, str]:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    cmd = [
        "ffmpeg",
        "-y",
        "-i",
        str(input_path),
        "-ar",
        "16000",
        "-ac",
        "1",
        "-sample_fmt",
        "s16",
        str(output_path),
    ]
    result = subprocess.run(
        cmd,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.PIPE,
        encoding="utf-8",
        errors="replace",
    )
    err = (result.stderr or "").strip()
    return result.returncode == 0, err


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument(
        "--public-key",
        default=DEFAULT_PUBLIC_KEY,
        help="Публичная ссылка Yandex Disk",
    )
    parser.add_argument("--zip-path", type=Path, default=DEFAULT_ZIP_PATH)
    parser.add_argument("--extract-dir", type=Path, default=DEFAULT_EXTRACT_DIR)
    parser.add_argument("--output-dir", type=Path, default=DEFAULT_OUTPUT_DIR)
    parser.add_argument("--log-file", type=Path, default=DEFAULT_LOG_FILE)
    parser.add_argument("--zip-out", type=Path, default=DEFAULT_ZIP_OUT)
    parser.add_argument(
        "--no-download",
        action="store_true",
        help="Не скачивать: ожидается готовый архив по --zip-path",
    )
    parser.add_argument(
        "--no-zip",
        action="store_true",
        help="Не собирать итоговый zip (только output-dir)",
    )
    parser.add_argument(
        "--no-colab-download",
        action="store_true",
        help="Не вызывать files.download (для запуска не в Colab)",
    )
    args, leftover = parser.parse_known_args()
    leftover = strip_leading_jupyter_kernel_args(leftover)
    if leftover:
        print(
            "Предупреждение: неизвестные аргументы (игнорируются): "
            + " ".join(leftover),
            file=sys.stderr,
        )

    extract_dir = args.extract_dir
    output_dir = args.output_dir
    zip_path = args.zip_path
    log_file = args.log_file
    zip_out = args.zip_out

    extract_dir.mkdir(parents=True, exist_ok=True)
    output_dir.mkdir(parents=True, exist_ok=True)

    if not args.no_download:
        print("Ссылка на скачивание с Yandex Disk…")
        download_yandex_public(args.public_key, zip_path)
    elif not zip_path.is_file():
        print(f"Ошибка: нет архива {zip_path} (используйте --no-download только если zip уже есть).", file=sys.stderr)
        raise SystemExit(1)

    print("Распаковка…")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(extract_dir)

    input_audio_dir = find_audios_folder(extract_dir)
    if input_audio_dir is None:
        raise FileNotFoundError("Папка «audios» не найдена в распакованном архиве.")

    print(f"Найдена папка audios: {input_audio_dir}")

    print("Копирование структуры…")
    shutil.copytree(extract_dir, output_dir, dirs_exist_ok=True)

    output_audio_dir = output_dir / input_audio_dir.relative_to(extract_dir)
    shutil.rmtree(output_audio_dir)
    output_audio_dir.mkdir(parents=True, exist_ok=True)

    input_files = [p for p in input_audio_dir.rglob("*") if p.is_file()]
    log_lines: list[str] = []
    converted = 0
    skipped_ext = 0
    failed = 0

    print("Конвертация аудио (ffmpeg)…")
    for file_path in tqdm(input_files, desc="Файлы"):
        if file_path.suffix.lower() not in SUPPORTED_EXTENSIONS:
            log_lines.append(f"SKIP EXT: {file_path}")
            skipped_ext += 1
            continue

        rel = file_path.relative_to(input_audio_dir)
        out_path = (output_audio_dir / rel).with_suffix(".wav")

        try:
            ok, err = convert_audio(file_path, out_path)
            if not ok:
                log_lines.append(f"FFMPEG FAILED: {file_path}\n{err[:2000]}")
                failed += 1
            else:
                converted += 1
        except Exception as e:
            log_lines.append(f"ERROR: {file_path}: {e}")
            failed += 1

    log_file.write_text("\n".join(log_lines) + ("\n" if log_lines else ""), encoding="utf-8")

    if not args.no_zip:
        print("Сборка zip…")
        shutil.make_archive(str(zip_out.with_suffix("")), "zip", output_dir)

    print("\nГотово")
    print("Сконвертировано:", converted)
    print("Пропущено (не то расширение):", skipped_ext)
    print("Ошибок:", failed)
    print("Лог:", log_file.resolve())

    if not args.no_colab_download and not args.no_zip:
        try:
            from google.colab import files  # type: ignore

            files.download(str(zip_out))
        except ImportError:
            print("(google.colab недоступен — скачивание через files.download пропущено.)")


if __name__ == "__main__":
    main()